# Fibromyalgia RAG Pipeline

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline for answering questions about fibromyalgia based on the research article:

*"Fibromyalgia: A Review of the Pathophysiological Mechanisms and Multidisciplinary Treatment Strategies"*

The pipeline includes the following stages:

1. Parsing
2. Cleaning
3. Chunking
4. Embedding and Vector Indexing
5. Retrieval Evaluation
6. Answer Generation

The system processes the article and retrieves relevant information to generate answers based on the document content.

## 1. Imports and Setup

In [6]:
%pip install -q pymupdf langchain-text-splitters tiktoken


In [7]:
import os
import re
import json
import bisect
import hashlib
from pathlib import Path
from collections import defaultdict

import fitz  # PyMuPDF


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. PDF Path

In [25]:
# Project-relative path (works locally, in Colab, and in CI alike)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Use the PDF from Google Drive when running in Colab
PDF_PATH = Path("/content/drive/MyDrive/biomedicines-12-01543.pdf")

assert PDF_PATH.exists(), (
    f"Source PDF not found at {PDF_PATH}."
)

print("Using PDF:", PDF_PATH)

Using PDF: /content/drive/MyDrive/biomedicines-12-01543.pdf


## 3. Parsing

The PDF is parsed using PyMuPDF to extract the document content while preserving its layout information.

The parsing process uses the `"dict"` output, which provides information about the text, including font properties and position. Numbered sections and subsections are identified based on their formatting.

Image blocks are excluded because this pipeline focuses on extracting and processing textual content.

### 3.1 Layout-aware extraction

In [26]:
JUNK_LINE_PATTERNS = [
    r'^Biomedicines\s+\d{4},\s*\d+,?\s*(x FOR PEER REVIEW|\d+)$',  # repeated citation banner
    r'^\d+\s+of\s+\d+$',                                             # "4 of 22" page markers
]

def extract_lines(pdf_path: Path):
    """Extract text lines with font metadata (layout-aware, not plain text)."""
    doc = fitz.open(pdf_path)
    lines = []
    block_id = 0
    for pno, page in enumerate(doc):
        d = page.get_text("dict")
        for block in d["blocks"]:
            if block["type"] != 0:  # skip images
                continue
            block_id += 1
            for line in block["lines"]:
                spans = line["spans"]
                if not spans:
                    continue
                text = "".join(s["text"] for s in spans).strip()
                if not text:
                    continue
                lines.append({
                    "page": pno + 1,
                    "block_id": block_id,
                    "text": text,
                    "fonts": {s["font"] for s in spans},
                    "y": line["bbox"][1],
                })
    doc.close()
    return [l for l in lines if not any(re.match(p, l["text"]) for p in JUNK_LINE_PATTERNS)]

lines = extract_lines(PDF_PATH)
print(f"Extracted {len(lines)} text lines (after removing running headers/footers)")
print(lines[5])


Extracted 1224 text lines (after removing running headers/footers)
{'page': 1, 'block_id': 6, 'text': 'Multidisciplinary Treatment', 'fonts': {'URWPalladioL-Roma'}, 'y': 528.604736328125}


### 3.2 Heading Detection

Headings are identified based on their numbering pattern and formatting:

| Level | Pattern | Formatting |
|---|---|---|
| 1 | `N. Title` | Bold |
| 2 | `N.N. Title` | Italic |
| 3 | `N.N.N. Title` | Regular text on a separate line |

In [27]:
H1 = re.compile(r'^(\d{1,2})\.\s+(.+)$')
H2 = re.compile(r'^(\d{1,2}\.\d{1,2})\.\s+(.+)$')
H3 = re.compile(r'^(\d{1,2}\.\d{1,2}\.\d{1,2})\.\s+(.+)$')

def _is_bold(block):
    return any('Bold' in f for f in block["fonts"])

def _is_italic(block):
    return any('Ital' in f for f in block["fonts"])

def detect_headings(lines):
    headings = []
    for i, b in enumerate(lines):
        t = b["text"]
        m3 = H3.match(t)
        m2 = H2.match(t) if not m3 else None
        m1 = H1.match(t) if not (m2 or m3) else None
        if m3:
            headings.append({"level": 3, "number": m3.group(1), "title": m3.group(2), "line_idx": i})
        elif m2 and _is_italic(b):
            headings.append({"level": 2, "number": m2.group(1), "title": m2.group(2), "line_idx": i})
        elif m1 and _is_bold(b):
            headings.append({"level": 1, "number": m1.group(1), "title": m1.group(2), "line_idx": i})
    return headings

headings = detect_headings(lines)
print(f"Detected {len(headings)} headings\n")
for h in headings:
    print(" " * ((h["level"] - 1) * 3), h["number"], h["title"])


Detected 22 headings

 1 Introduction
 2 Epidemiology
 3 Physiopathology
    3.1 Underlying Processes in Fibromyalgia
       3.1.1 Central Sensitization
       3.1.2 Peripheral Sensitization
       3.1.3 Inflammation
 4 Etiopathogenesis
 5 Diagnosis
    5.1 Bases and Diagnostic Advances
    5.2 Diagnostic Biomarkers
       5.2.1 Genetic Biomarkers
       5.2.2 Serological Biomarkers
       5.2.3 Role of Vibrational Spectroscopy in Fibromyalgia
 6 Treatment
    6.1 Pharmacological Treatment
    6.2 Non-Pharmacological Treatments: Physical Therapy Treatment
       6.2.1 Exercise Therapy
       6.2.2 Hydrotherapy
       6.2.3 Electrotherapy
       6.2.4 Manual Therapy
 7 Conclusions


### 3.3 Section building

Slice the line stream between consecutive headings, rejoining hyphenated words at the
point where the original `-` + line-break pattern is still visible. Each resulting
element keeps the structural metadata needed downstream: **page number, section,
subsection**.

In [28]:
def join_lines_dehyphenated(text_lines) -> str:
    out = ""
    for line_dict in text_lines:
        line = line_dict["text"]
        if out.endswith("-") and line and line[0].islower():
            out = out[:-1] + line          # rejoin split word, no space
        elif out:
            out = out + " " + line
        else:
            out = line
    return out

def build_sections(lines, headings):
    elements = []

    current_h1 = "Unknown Section"
    current_sub = "Unknown Subsection"

    for i, h in enumerate(headings):
        if h["level"] == 1:
            current_h1 = f"{h['number']} {h['title']}"
            current_sub = current_h1
        else:
            current_sub = f"{h['number']} {h['title']}"

        start = h["line_idx"] + 1
        end = headings[i + 1]["line_idx"] if i + 1 < len(headings) else len(lines)

        current_block_lines = []
        current_block_id = None

        for line in lines[start:end]:
            if current_block_id is None:
                current_block_id = line["block_id"]

            if line["block_id"] != current_block_id:
                if current_block_lines:
                    text = re.sub(r'\s+', ' ', join_lines_dehyphenated(current_block_lines)).strip()
                    if text:
                        elements.append({
                            "source": PDF_PATH.name,
                            "page_number": current_block_lines[0]["page"],
                            "section": current_h1,
                            "subsection": current_sub,
                            "text": text
                        })
                current_block_lines = [line]
                current_block_id = line["block_id"]
            else:
                current_block_lines.append(line)

        if current_block_lines:
            text = re.sub(r'\s+', ' ', join_lines_dehyphenated(current_block_lines)).strip()
            if text:
                elements.append({
                    "source": PDF_PATH.name,
                    "page_number": current_block_lines[0]["page"],
                    "section": current_h1,
                    "subsection": current_sub,
                    "text": text
                })

    return elements

parsed_elements = build_sections(lines, headings)
for e in parsed_elements[:3]:
    print(f"[{e['page_number']}] {e['section']} -> {e['subsection']}: {e['text'][:50]}...")
print(f"\nTotal elements (paragraphs): {len(parsed_elements)}")


[1] 1 Introduction -> 1 Introduction: Fibromyalgia is a chronic functional pathology cha...
[1] 1 Introduction -> 1 Introduction: Biomedicines 2024, 12, 1543. https://doi.org/10.33...
[2] 1 Introduction -> 1 Introduction: in most patients, its onset may be associated with...

Total elements (paragraphs): 82


### 3.4 Reference-list parsing

Parsed into individually addressable, numbered entries so an in-text marker like
`[12,45]` can be resolved back to its source. Kept as-is from the source notebook;
not required by Cleaning or Chunking below, but preserved since it's part of the
existing working parsing logic.

In [29]:
REF_ENTRY = re.compile(r'\n(\d{1,3})\.\s+(?=[A-Za-z])')

def parse_references(pdf_path: Path) -> dict:
    doc = fitz.open(pdf_path)
    raw_text = "".join(page.get_text() + "\n" for page in doc)
    doc.close()

    m = re.search(r'\nReferences\n', raw_text)
    if not m:
        return {}
    ref_text = raw_text[m.end():]
    ref_text = ref_text.split("Disclaimer/Publisher")[0]
    ref_text = re.sub(r'Biomedicines\s+\d{4},\s*\d+,?\s*\d+\s*\n?\d*\s*of\s*\d+\s*\n?', '', ref_text)
    ref_text = re.sub(r'\n\d+\s+of\s+\d+\n', '\n', ref_text)

    matches = list(REF_ENTRY.finditer("\n" + ref_text))
    entries = {}
    for i, mm in enumerate(matches):
        num = int(mm.group(1))
        start = mm.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(ref_text) + 1
        content = ("\n" + ref_text)[start:end]
        entries[num] = re.sub(r'\s+', ' ', content).strip()
    return entries

references = parse_references(PDF_PATH)
print(f"Parsed {len(references)} reference entries (expected 148)")
missing = set(range(1, 149)) - set(references)
print("Missing entry numbers:", missing or "none")


Parsed 148 reference entries (expected 148)
Missing entry numbers: none


## 4. Cleaning


In [30]:
def clean_element_text(text: str) -> str:
    cleaned = text
    # Re-join words hyphenated across a line break (safety net; Parsing already does this)
    cleaned = re.sub(r'-\s*\n\s*', '', cleaned)
    # Remove the repeated journal header/footer boilerplate
    cleaned = re.sub(r'Biomedicines\s+2024,\s*12,\s*1543\.?', '', cleaned)
    # Remove DOI / journal URL boilerplate (article banner, not reference-list DOIs)
    cleaned = re.sub(r'https://doi\.org/10\.3390/biomedicines\d+', '', cleaned)
    cleaned = re.sub(r'https://www\.mdpi\.com/journal/biomedicines', '', cleaned)
    # Remove leftover "N of 22" page markers (extraction noise)
    cleaned = re.sub(r'\b\d+\s+of\s+22\b', '', cleaned)
    # Collapse whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

cleaned_elements = []
for el in parsed_elements:
    text = clean_element_text(el["text"])
    if text:  # drop elements that were pure boilerplate and are now empty
        cleaned_elements.append({**el, "text": text})

print(f"Cleaned elements: {len(cleaned_elements)} (from {len(parsed_elements)} parsed)")
for e in cleaned_elements[:3]:
    print(f"[{e['page_number']}] {e['section']} -> {e['subsection']}: {e['text'][:80]}...")


Cleaned elements: 81 (from 82 parsed)
[1] 1 Introduction -> 1 Introduction: Fibromyalgia is a chronic functional pathology characterized by widespread muscu...
[2] 1 Introduction -> 1 Introduction: in most patients, its onset may be associated with specific conditions such as i...
[2] 2 Epidemiology -> 2 Epidemiology: Fibromyalgia is a highly prevalent syndrome in the general population, being con...


## 5. Chunking



In [31]:
%pip install -q langchain-text-splitters tiktoken


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

elements = cleaned_elements  # in-memory hand-off from Cleaning

print(f"Loaded {len(elements)} cleaned elements.")


Loaded 81 cleaned elements.


### 5.1 Group by subsection

In [33]:
# Group elements by (section, subsection)
subsections_map = defaultdict(list)
for el in elements:
    key = (el["section"], el["subsection"])
    subsections_map[key].append(el)

print(f"Found {len(subsections_map)} unique subsections.")


Found 20 unique subsections.


### 5.2 Structure-aware chunking

For each subsection, combine its paragraph elements into a continuous string so
`RecursiveCharacterTextSplitter` can do its job efficiently. To preserve the accurate
`page_number` of every chunk without relying on text string markers, build a mapping
between the character index and the original page number, then use the chunk's start
index to look up its exact page.

In [34]:
_encoder = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(_encoder.encode(text))

# Separators demote commas so we don't inappropriately split scientific sentences early.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
    length_function=token_len,
    separators=[
        "\n\n", # paragraph boundary
        "\n",   # line boundary
        ". ",   # sentence boundary
        "? ",
        "! ",
        "; ",
        " ",
        "",
    ],
)

final_chunks = []
section_counters = {}

for (section, subsection), sub_elements in subsections_map.items():
    # 1. Combine subsection elements into a continuous string, keeping a page_map
    combined_text = ""
    # page_map is a list of tuples: (char_start_index, page_number)
    page_map = []

    for el in sub_elements:
        page_map.append((len(combined_text), el["page_number"]))
        combined_text += el["text"] + "\n\n"

    # 2. Split the continuous string into chunks
    chunk_texts = text_splitter.split_text(combined_text)

    # 3. Create metadata for each chunk
    search_start = 0
    section_counters[section] = section_counters.get(section, 0)

    for chunk_text in chunk_texts:
        section_counters[section] += 1

        # Find the starting character index of this chunk in the combined text
        # We use a tracking index (search_start) to handle overlapping/duplicate text
        chunk_start_idx = combined_text.find(chunk_text, search_start)
        if chunk_start_idx != -1:
            search_start = chunk_start_idx + 1 # advance for the next find
        else:
            chunk_start_idx = search_start # fallback

        chunk_end_idx = chunk_start_idx + len(chunk_text)

        # Find the corresponding page numbers using bisect on the page_map
        mapping_start_idx = bisect.bisect_right([m[0] for m in page_map], chunk_start_idx) - 1
        mapping_start_idx = max(0, mapping_start_idx)

        mapping_end_idx = bisect.bisect_right([m[0] for m in page_map], chunk_end_idx) - 1
        mapping_end_idx = max(0, mapping_end_idx)

        page_numbers = []
        for m_idx in range(mapping_start_idx, mapping_end_idx + 1):
            page_numbers.append(page_map[m_idx][1])

        # Deduplicate and sort
        page_numbers = sorted(list(set(page_numbers)))

        # Create a robust, unique chunk ID
        source = sub_elements[0]["source"]
        chunk_index = section_counters[section]
        hash_input = f"{source}_{section}_{subsection}_{chunk_index}"
        chunk_id = hashlib.md5(hash_input.encode("utf-8")).hexdigest()[:12]

        final_chunks.append({
            "source": source,
            "page_numbers": page_numbers,
            "section": section,
            "subsection": subsection,
            "chunk_id": chunk_id,
            "chunk_index": chunk_index,
            "n_tokens": token_len(chunk_text),
            "n_chars": len(chunk_text),
            "text": chunk_text
        })

print(f"Generated {len(final_chunks)} chunks.")


Generated 98 chunks.


### 5.3 Filter meaningless small chunks

Drop empty, extraction-noise, or meaningless fragments. Small chunks are retained if
they contain meaningful alphabetic content (like a heading or brief statement).

In [35]:
def is_meaningful(chunk):
    if chunk["n_tokens"] < 5:
        return False
    # Require at least some alphabetical characters to avoid dropping just numbers/punctuation noise
    if not re.search(r'[a-zA-Z]{3,}', chunk["text"]):
        return False
    return True

filtered_chunks = [c for c in final_chunks if is_meaningful(c)]

print(f"Final chunk count after filtering: {len(filtered_chunks)}")


Final chunk count after filtering: 98


## 6. Embedding & Indexing

Embed every chunk with a small sentence-transformer model and index the
vectors in a FAISS store for similarity search.

In [36]:
!pip install -q langchain-community faiss-cpu openai

In [39]:
import os
from pathlib import Path
from openai import OpenAI
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# API key
os.environ["OPENROUTER_API_KEY"] = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

class OpenRouterEmbeddings(Embeddings):
    def embed_documents(self, texts):
        response = client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=texts,
            encoding_format="float"
        )
        return [item.embedding for item in response.data]

    def embed_query(self, text):
        response = client.embeddings.create(
            model="nvidia/nemotron-3-embed-1b:free",
            input=text,
            encoding_format="float"
        )
        return response.data[0].embedding

embeddings = OpenRouterEmbeddings()

# Convert final_chunks from dictionaries to LangChain Documents
chunks = [
    Document(
        page_content=chunk["text"],
        metadata={
            "source": chunk["source"],
            "page_numbers": chunk["page_numbers"],
            "section": chunk["section"],
            "subsection": chunk["subsection"],
            "chunk_id": chunk["chunk_id"],
            "chunk_index": chunk["chunk_index"],
            "n_tokens": chunk["n_tokens"],
            "n_chars": chunk["n_chars"]
        }
    )
    for chunk in final_chunks
]

# Create FAISS vector store
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Indexing complete:", vectorstore.index.ntotal, "vectors")

# Save FAISS index in Google Drive
INDEX_DIR = Path("/content/drive/MyDrive/faiss_index")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

vectorstore.save_local(str(INDEX_DIR))

print("Saved FAISS index to", INDEX_DIR)

Indexing complete: 98 vectors
Saved FAISS index to /content/drive/MyDrive/faiss_index


## 7. Retrieval Evaluation

A lightweight Precision@K benchmark: for each test question, check whether
any of the top-K retrieved chunks contain at least one of the expected
keywords.

In [40]:
def evaluate_retrieval(eval_dataset, retriever) -> float:
    relevant_count = 0

    for item in eval_dataset:
        docs = retriever.invoke(item["question"])
        retrieved_text = " ".join(d.page_content for d in docs)

        if any(kw.lower() in retrieved_text.lower() for kw in item["keywords"]):
            relevant_count += 1

    score = (relevant_count / len(eval_dataset)) * 100

    print(f"Retrieval Precision@K Score: {score:.2f}%")

    return score


eval_dataset = [
    {
        "question": "What are the FDA-approved drugs for fibromyalgia?",
        "keywords": ["pregabalin", "duloxetine", "milnacipran"],
    },
    {
        "question": "What is fibromyalgia characterized by?",
        "keywords": ["chronic", "widespread", "pain"],
    },
    {
        "question": "What diagnostic tools or criteria are mentioned?",
        "keywords": ["WPI", "SS scale", "ACR"],
    },
]

precision_at_k = evaluate_retrieval(eval_dataset, retriever)

Retrieval Precision@K Score: 100.00%
